## PHYS-467 Machine Learning for Physicists. Exercise session 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso
import sklearn.linear_model as linear_model
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import sklearn.preprocessing as preprocessing

# Exercise 1: Linear Regression in 1D

The following function creates one dimensional linear data and add a gaussian noise i.e $y = a \times x + \varepsilon$

In [ ]:
def generate_data(n_data, coefficient, variance):
    X = np.random.normal(size=(n_data,))
    y = X * coefficient + np.sqrt(variance) * np.random.normal(size=(n_data,))

    return X.reshape((-1, 1)), y

We can generate and plot some data (X, Y), along with the noiseless function $y = a \times x$

In [ ]:
coefficient = 3
n_data      = 10
X, y = generate_data(n_data, coefficient, 0.5)
X.shape

In [ ]:
# Plot the training data

plt.scatter(X, y)
plt.xlabel('X')
plt.ylabel('y');

# ground truth 
plt.plot(X, coefficient * X, c='k')

**Question 1** Write a function to compute the least-square regression using the closed form seen in class.
Use this function on the  training data generated previously.

In [ ]:
# Write a function to compute the linear-regression predictor in close form

def linear_regression(X, y):
    """
    arguments:
        - X : data matrix
        - y : output
    returns:
        - w : the least square estimator
    """
    ### your code here
    return w

In [ ]:
# Print the predicted weights (coefficient + bias)

w = linear_regression(X, y)
w

**Question 3** Generate a grid of points and apply your predictor on this grid. Plot it along your training set.

In [ ]:
# Plot the training points and the predictor evaluated on a set of test points

X_test = np.linspace(np.min(X), np.max(X), 10)
y_test_pred = ### your code here

plt.scatter(X, y)
plt.plot(X_test, coefficient * X_test, 'k')
plt.plot(X_test, y_test_pred, 'r--')
plt.xlabel('X')
plt.ylabel('y');

**Question 4** Repeat the same with `LinearRegression` of the module [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html). Check that the coefficients are the same.

In [ ]:
# Repeat with Scikit-learn

linear_regression_sk = LinearRegression()
### your code here. If at loss, google the documentation!
print('sklearn : ', linear_regression_sk.coef_, linear_regression_sk.intercept_)
print('w : ', w)

# Exercise 2: Polynomial Regression, Underfitting & Overfitting

We first generate _noiseless_ points $x_1, ... x_n$ with few samples (say $5 \leqslant n \leqslant 10$) and $y_i = x_i^2$. Using [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) from sklearn, we can fit polynomials of varying degrees $d$ on your training data.


In [ ]:
def generate_data(polynome, bruit_std, n):
    """
    arguments:
        - polynome  : list of coefficients representing the polynom (polynome[i] multiplies X^i)
        - bruit_std : standard deviation of the gaussian noise
        - n         : number of data points
    returns: 
        - (X, y)    : list of n points with y = P(x) + gaussian noise
    """    
    degree  = len(polynome) - 1
    samples = np.random.uniform(-2, 2, size=n)
    xs      = np.copy(samples)
    samples = np.ones((n, degree + 1)).T * samples
    powers  = np.power(samples.T, np.arange(0, degree + 1, 1))
    return xs, np.sum(powers * polynome, axis = 1) + np.random.normal(0, bruit_std, size=n)

In [ ]:
n       = 5
bound   = 3
# function y = x^2
polynom = np.array( [0, 0, 1] )

xs, ys   = generate_data(polynom, 0.0, n)

Here is how you fit the data : you first preprocess it (create the polynomial features) xs_trans, then do a linear regression.

In [ ]:
# Fit the data

x_grid = np.linspace(-bound, bound, 50)

degrees = np.arange(1, 10, 2)
coefs = []


for d in degrees:
    poly     = preprocessing.PolynomialFeatures(d)
    xs_trans = poly.fit_transform(xs.reshape(-1, 1))
    x_grid_trans = poly.fit_transform(x_grid.reshape((-1, 1)))
    lr = linear_model.Ridge(alpha = 0.0, fit_intercept=False).fit(xs_trans, ys)
    y_pred = xs_trans @ lr.coef_
    coefs.append(lr.coef_)
    plt.plot(x_grid, x_grid_trans @ lr.coef_, label=f'degree = {d}')
    print(f'Degree {d} : training error is {mean_squared_error(ys, y_pred)}')

plt.scatter(xs, ys, marker='.', c='k')
plt.ylim(np.min(ys) - 10, np.max(ys) + 10)
plt.legend()

**Question 1** Now, we generate _noisy_ quadratic data (i.e $y = x^2 + \varepsilon$). Fit again polynoms of various degrees. Sample a large test set and plot the test error as a function of the degree $d$ of the predictor. Observe the under/overfitting phenomenon.

Add a $\ell_2$ regularisation term for the high-degree polynomial case. What happens ? Try to vary the strenth of the regularization.

In [ ]:
xs, ys  = generate_data(polynom, 0.5, 10)
y_true  = x_grid**2

degrees = np.array([1, 2, 4, 8])

plt.scatter(xs, ys, c='k', marker='.')

for d in degrees:
    poly     = ## your code here
    xs_trans = ## your code here
    x_grid_trans = poly.fit_transform(x_grid.reshape((-1, 1)))
    lr = ## your code here
    y_pred = ## your code here
    plt.plot(x_grid, y_pred, label=f'degree = {d}')
    print(f'Degree {d} : MSE is {mean_squared_error(y_true, y_pred)}')

plt.legend()
plt.ylim(min(ys), max(ys))
plt.show()

In [ ]:
lambda_range = np.logspace(-2, 6, 50)
d = 6
ridge_mses = []

poly     = preprocessing.PolynomialFeatures(d)
xs_trans = poly.fit_transform(xs.reshape(-1, 1))
x_grid_trans = poly.fit_transform(x_grid.reshape((-1, 1)))
y_true = x_grid**2

for lambda_ in lambda_range:
    lr = linear_model.Ridge(alpha = lambda_, fit_intercept=False).fit(xs_trans, ys)
    y_pred = x_grid_trans @ lr.coef_
    ridge_mses.append(mean_squared_error(y_true, y_pred))

plt.semilogx(lambda_range, ridge_mses)


# Exercise 3: Real data

The superconductivity dataset [1] contains 81 chemical and molecular features extracted from 21263 superconductors along with the critical temperature (the label) in the 82nd column. The goal is to predict the critical temperature based on the 81 first features. If you are interested by the physical meaning of those 81 features, check out the original paper at https://arxiv.org/pdf/1803.10260.pdf !

[1] Hamidieh, Computational Materials Science, 2018

First, you need to download the dataset, in .csv format. You can find it here : https://archive.ics.uci.edu/dataset/464/superconductivty+data or on the Moodle.

Now, load the data. Since the format is .csv (data table), we used the Python module called pandas. If you are locally running the notebook, you need to indicate the path to it, e.g. "Downloads/train.csv". If you use colab, you have to upload it to colab (see the left hand side icon).

In [ ]:
X=pd.read_csv("../super/train.csv",index_col=None)

using .values() converts the table into a numpy array . Remember, the last column is the labels.

In [ ]:
X_tot=X.values[:,:-1]
y_tot=X.values[:,-1]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_tot, y_tot, test_size=0.2)
X_train, X_validation, y_train, y_validation = train_test_split(X_train, y_train, test_size=0.25)

Because different features correspond to different physical quantities and have different units, thus different magnitudes of numerical values, we preprocess the array.

In [ ]:
scaler_X = preprocessing.StandardScaler().fit(X_train)
scaler_y = preprocessing.StandardScaler().fit(y_train.reshape(-1,1))

In [ ]:
X_test=scaler_X.transform(X_test)
X_train=scaler_X.transform(X_train)
X_validation=scaler_X.transform(X_validation)


**Question 1** Fit a linear model on the training set, and compute the predictions $\hat y$ of your model. Compare them to the true values $y^*$, for example by printing the mean squared error.

In [ ]:
# Perform linear regression on the training set

linear_regression_sk = LinearRegression()
## your code here

In [ ]:
# Print the test error and plot the predicted prices vs the actual ones

y_pred = ## your code here
print(f'mse: {mean_squared_error(y_test, y_pred)}')

plt.plot(np.linspace(np.min(y_test),np.max(y_test),100),np.linspace(np.min(y_test),np.max(y_test),100),ls="--",c="r")

plt.scatter(y_test, y_pred)
plt.xlabel('true Tc')
plt.ylabel('predicted Tc');

**Question 3** Redo the question 2 while varying the size of the training set. For each size, compute the training and test error. How do they vary as a function of the training size ? 

In [ ]:
# Vary the training set size n and plot the learning curves (test and train error vs n)

trainset_sizes = [10, 20, 30, 50, 100, 200, 300]

train_errors = []
test_errors = []

for n in trainset_sizes:

    ## your code here

    y_train_pred = ## your code here
    train_errors.append(mean_squared_error(y_train[:n], y_train_pred))

    y_test_pred = ## your code here
    test_errors.append(mean_squared_error(y_test, y_test_pred))


plt.plot(trainset_sizes, test_errors, label='Test error')
plt.ylim(0,10000)
plt.xlabel('training set size')
plt.legend();

**Question 4** The function below generates _non-interactive_ polynomial features : for a degree $k$ and an input $x = x_1, ..., x_d$, generate $\tilde x = x_1, ..., x_1^k, x_2, ..., x_2^k, ... x_d^k$.
Run a linear regression on these polynomial features with different degrees $k$. 
Observe the under/overfitting phenomenon.

In [ ]:
# Implement *non-interacting* polynomial-features regression (not present in sklearn, write a function using numpy) and vary the degree from 1 to 6. Plot the results.


def non_interacting_polyfeat(X, k):
    """
    arguments: 
        - X : n x d matrix, where each row is a data sample
        - k : degree of the desired polynomial features
    returns: 
        - X2 : n x (dk) matrix containing the polynomial features
    """
    return np.concatenate([X**(i+1) for i in range(k)], axis=1)

train_errors = []
val_errors = []

ndegs = 10
for degree in range(1, ndegs):

    X_train_poly = non_interacting_polyfeat(X_train, degree)
    linear_regression_sk.fit(X_train_poly, y_train)
    y_train_pred_poly = ## your code here
    train_errors.append(mean_squared_error(y_train, y_train_pred_poly))

    y_val_pred_poly = ## your code here
    val_errors.append(mean_squared_error(y_validation, y_val_pred_poly))

plt.semilogy(np.arange(1,ndegs), train_errors, label='Train error')
plt.semilogy(np.arange(1,ndegs), val_errors, label='Validation error')
plt.xlabel('Degree (model complexity)')
plt.legend();

**Question 5** From the previous question, pick a degree that you think is best, and add different regularisation strength $\lambda$ to the regression. Plot the training and test errors as a function of $\lambda$.

In [ ]:
# Add different regularisers to the degree-6 model. Plot the results and find the best value.

best_degree = list(range(1, ndegs))[np.argmin(val_errors)]
print('Best degree is ', best_degree)

reg_strenghts = np.logspace(-12, 6, 20)

train_errors = []
val_errors_lambda = []

for reg_strenght in reg_strenghts:

    X_train_poly = non_interacting_polyfeat(X_train, k=best_degree)

    ridge_regression = ## your code here
    ## your code here
    
    y_train_pred_poly = ridge_regression.predict(X_train_poly)
    train_errors.append(mean_squared_error(y_train, y_train_pred_poly))

    y_val_pred_poly = ridge_regression.predict(non_interacting_polyfeat(X_validation, k=best_degree))
    val_errors_lambda.append(mean_squared_error(y_validation, y_val_pred_poly))

plt.loglog(reg_strenghts, train_errors, label='Train error')
plt.loglog(reg_strenghts, val_errors_lambda, label='Validation error')
plt.xlabel('$\lambda$')
plt.legend();

In [ ]:
lambda_opt = reg_strenghts[np.argmin(val_errors_lambda)]
print(f'Optimal lambda is {lambda_opt}, the validation error is {np.min(val_errors_lambda)}')

ridge_regression = Ridge(alpha=lambda_opt)
ridge_regression.fit(X_train_poly, y_train)

y_test_pred_poly = ridge_regression.predict(non_interacting_polyfeat(X_test, k=best_degree))
test_error = mean_squared_error(y_test, y_test_pred_poly)
print(f'Test error is {test_error}')

# Bonus for the curious: Spectrum of covariance in high dimensions

**Question 1** Consider $X$ a $n \times d$ Gaussian matrix. Compute and plot the spectrum of the covariance matrix $X^T X$ for dimension d = 50 and varying $n$. Look at the evolution of its rank as a function of $\alpha = n/d$. What happens at $\alpha = 1$ ?

In [ ]:
def generate_covariance_matrix(n, d):
    """
    arguments: 
        - n : number of samples
        - d : dimension
    returns:
        - covariance matrix of n x d matrix X
    """
    X = np.random.normal(size=(n, d))
    return X.T @ X / n

In [ ]:
# Compute the rank and plot the eigenvalues of the matrix X.T X varying alpha = n / d

d = 200
n_list = np.arange(int(0.1 * d), int(2 * d), 20)

rank_list = []

for n in n_list:

    cov = generate_covariance_matrix(n, d)
    rank = np.linalg.matrix_rank(cov)
    rank_list.append(rank)
    evals = np.linalg.eigh(cov)[0]
    evals = np.sort(evals)[::-1]

    plt.semilogy(evals, 'o')
    plt.xlabel(r'$\rho$')
    plt.ylabel(r'$\eta_{\rho}$')
    plt.title(f'$\\alpha = {n / d}')

    plt.show()

In [ ]:
plt.plot(n_list, rank_list, marker='.')
plt.show()

Question 3 Plot the behaviour of the test error vs alpha for noiseless high-dimensional linear data in the absence of noise. You can use  𝑑=100
  and  10⩽𝑛⩽500
 . What do you observe at  𝛼=1

In [ ]:
from sklearn.metrics import mean_squared_error

d = 100

train_set_sizes = np.array([500, 300, 200, 100, 50, 30, 20, 10])

X = np.random.rand(np.max(train_set_sizes) + 500, d)
w = 2 * np.random.randn(d, 1)
y = X.dot(w) + 2

mse = []
for n in train_set_sizes:
    X_train = X[:n]
    y_train = y[:n]
    linear_regression_sk.fit(X_train, y_train) # Fits the training data
    X_test = X[-500:]
    y_test = y[-500:]
    y_test_pred = linear_regression_sk.predict(X_test) # Predicts the labels of the test points
    mse.append(mean_squared_error(y_test, y_test_pred)) # Computes the mean squared error

plt.semilogx(train_set_sizes / d, mse, 'o')
plt.xlabel(r'$\alpha=\frac{n}{d}$')
plt.ylabel('Test error');